In [ ]:
import gpytoolbox as gpy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.collections as mc
import matplotlib.patches as mp
import trimesh

import dvx_ext

# 3D Voxelization Functions Overview

## Helper Functions

In [ ]:
mesh = trimesh.load_mesh("hand.obj")
V, F = np.array(mesh.vertices, dtype=np.float32), np.array(mesh.faces, dtype=np.int32)

# Global parameters
GRID_SIZE = (512, 512, 512)
FILTER_RADIUS = (2 / GRID_SIZE[0]) / 2
NUM_SAMPLES_PER_VOXEL = 64
NUM_SAMPLES_PER_SIMPLEX = 64

print(f"Mesh info: {len(V)} vertices, {len(F)} faces")
print(f"Grid size: {GRID_SIZE}")
print(f"Filter radius: {FILTER_RADIUS}")

In [ ]:
def primal(func, dtype, **kwargs):
    v = V.astype(dtype)
    f = F.astype(np.uint32)
    occupancy = np.zeros(GRID_SIZE, dtype=dtype)
    result = %timeit -o func(v, f, occupancy, **kwargs)
    return result


def forward_ad(forward_func, dtype, **kwargs):
    v = V.astype(dtype)
    f = F.astype(np.uint32)
    
    occupancy = np.zeros(GRID_SIZE, dtype=dtype)

    # Random perturbation in vertex space
    d_vertices = np.random.randn(*v.shape).astype(dtype) * 0.01
    
    # Forward AD
    d_occupancy_ad = np.zeros(GRID_SIZE, dtype=dtype)
    result = %timeit -o forward_func(v, f, occupancy, d_vertices, d_occupancy_ad, **kwargs)

    return result


def backward_ad(backward_func, dtype, **kwargs):
    v = V.astype(dtype)
    f = F.astype(np.uint32)
    
    occupancy = np.zeros(GRID_SIZE, dtype=dtype)
    d_vertices_ad = np.zeros_like(v)

    d_L = np.random.randn(*GRID_SIZE).astype(dtype)

    # Backward AD
    result = %timeit -o backward_func(v, f, occupancy, d_vertices_ad, d_L, **kwargs)

    return result


def test(ToTestFunctions, ToTestMethod, ToTestDtype):
    primal_kwargs = {'num_samples_per_voxel': NUM_SAMPLES_PER_VOXEL, 'filter_radius': FILTER_RADIUS}
    ad_kwargs = {'num_samples_per_simplex': NUM_SAMPLES_PER_SIMPLEX, 'filter_radius': FILTER_RADIUS}
    speed_results = {}

    for func in ToTestFunctions:
        for method in ToTestMethod:
            for dtype in ToTestDtype:
                print(f"Testing {func.__name__} with method={method}, dtype={dtype}")
                # Set kwargs
                if func == primal:
                    direction = ''
                    kwargs = primal_kwargs if method == 'mc' else {}
                elif func == forward_ad:
                    direction = '_forward'
                    kwargs = ad_kwargs if method == 'mc' else {}
                elif func == backward_ad:
                    direction = '_backward'
                    kwargs = ad_kwargs if method == 'mc' else {}
                
                if dtype == np.float64:
                    dt = 'f64'
                else:
                    dt = 'f32'

                test_func = getattr(dvx_ext, f'voxelize{direction}_{method}_{dt}')
                # Run test
                result = func(test_func, dtype, **kwargs)
                speed_results[test_func.__name__] = result
    return speed_results

In [ ]:
ToTestFunctions = [ primal ] #, forward_ad, backward_ad]
ToTestMethod = ['cf' ] # , 'mc']
ToTestDtype = [np.float32, np.float64]

In [ ]:
results = test(ToTestFunctions, ToTestMethod, ToTestDtype)

In [ ]:
for key, value in results.items():
    print(f"{key}: {value}")